# SpaceJEPA v3 · 1. Train v2 and v1, five seeds each

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rk0604/ReEntryAI/blob/jepa-v3/StochasticEntrySim/ReEntryAI_JEPA_v3_1_train.ipynb)

Same 6,000 flights, one fixed split (episode seed 0, 80/10/10), normalization from the training split.

| | v2 | v1 (fair budget) |
|---|---|---|
| model | 5-member ensemble, 12.3M parameters | 26k-parameter TCN + GRU (+ 6k prober), one model |
| stage 1 | 24,000 steps × 256 windows/member, horizon 24 → 48 → 96 | **24,000 steps × 256**, T = 24 (v1-wide in v2 had 6,000) |
| stage 2 | 6,000 × 512, horizon 16 → 32 → 48 | 6,000 × 512, T = 16 |
| SIGReg | per-step, **λ = 0.08 fixed** (v2's validation sweep) | pooled, λ = 0.02 |

**Seed s** sets the weight init, the data order and window sampling, and the SIGReg projections. v1 also
follows the plan's **convergence rule**: if its validation loss still improved by more than 2% over the
last 20% of training, it trains once more for 50% of the steps, with a warm restart at 30% of the peak
learning rate. The extension is recorded in the checkpoint.

**Runtime** (measured in v2's run): v2 ≈ 50 min per seed on an **L4**, ≈ 1.6 h on a **T4**; v1 ≈ 5–10
min per seed. All five seeds: **L4 ≈ 4.5 h, T4 ≈ 8.5 h**. Checkpoints are written to Drive every 10 min,
so the work can span several sessions: re-run all cells and it resumes. Each finished seed is
`02_models/<v1|v2>/seed_<k>/ckpt.pt`, with its training log and curves.

## 0. Setup

On Colab this clones the `jepa-v3` branch, installs `pymsis`, and mounts Google Drive. Every output
goes to **`MyDrive/ReEntryAI_jepa_v3/`** (`ReEntryAI_jepa_v3_smoke/` in smoke mode); the layout is in
`jepa/v3_plan.md` §9 and in the auto-written `00_study/README.md`. The 6,000 flights are read from
`MyDrive/ReEntryAI_jepa_data/wide_6000/`. Every stage is resumable: after a disconnect, run all
cells again and finished work is skipped.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "jepa-v3"
MODE = os.environ.get("JEPA_V3_MODE", "full")     # "full": the study | "smoke": every stage, tiny, numbers meaningless
if IN_COLAB:
    REPO = "/content/ReEntryAI"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/rk0604/ReEntryAI.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q pymsis
    os.chdir(f"{REPO}/StochasticEntrySim")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    STUDY_ROOT = f"{DRIVE}/ReEntryAI_jepa_v3" + ("" if MODE == "full" else f"_{MODE}")
    DATA_CANDIDATES = [f"{DRIVE}/ReEntryAI_jepa_data/wide_6000/episodes_wide_6000.pkl",
                       f"{DRIVE}/ReEntryAI_jepa_data/wide_6000"]
    LOCAL_DATA = "/content/data"
else:
    STUDY_ROOT = os.environ.get("JEPA_V3_ROOT", f"jepa_runs/v3_{MODE}")
    DATA_CANDIDATES = [os.environ.get("JEPA_WIDE_DATA", ""), "jepa_runs/episodes_wide_6000.pkl", "jepa_runs/wide_6000"]
    LOCAL_DATA = "jepa_runs"

import time
from pathlib import Path
SIM_DIR = Path.cwd()
assert (SIM_DIR / "rl_env.py").exists(), "start Jupyter in StochasticEntrySim/"
sys.path.insert(0, str(SIM_DIR))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline
from jepa import testset_v3 as TS, study_v3 as S, fly_v3 as F, analyze_v3 as A

CFG = S.StudyConfig.smoke() if MODE == "smoke" else S.StudyConfig()
L = S.Layout(STUDY_ROOT).make()
FROZEN = S.freeze_config(CFG, L)
CPUS = os.cpu_count() or 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MODE={MODE}  study folder={STUDY_ROOT}  device={DEVICE}  cpus={CPUS}  git={FROZEN['git_commit'][:10]}")

In [ ]:
from jepa import models_v2 as V, models as M1, train_v2 as T2
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"GPU {torch.cuda.get_device_name(0)} ({GPU_MEM_GB:.0f} GB), AMP {V.amp_dtype(torch.device('cuda'))}")
elif MODE == "full":
    print("WARNING: no GPU. Runtime -> Change runtime type -> L4 or T4.")
GRAD_CKPT = DEVICE == "cuda" and GPU_MEM_GB < 20      # T4 (15 GB): recompute activations to fit T = 96
torch.set_num_threads(1 if DEVICE == "cuda" else max(1, CPUS // 2))
SEEDS_THIS_SESSION = list(CFG.seeds)                  # e.g. [0, 1] to split the work over sessions

## 1. Data

In [ ]:
t0 = time.time()
src = T2.find_wide_dataset(DATA_CANDIDATES, download_dir=LOCAL_DATA)
DATA_PATH = T2.local_copy(src, LOCAL_DATA)
data = S.load_data(CFG, DATA_PATH, DEVICE)
print(f"{data['n_episodes']} flights from {src}: train/val/test = {len(data['train'])}/{len(data['val'])}/"
      f"{len(data['test'])} episodes, {len(data['ds_tr']):,} v2 training windows ({time.time() - t0:.0f}s)")
wm_ = V.WorldModelV2(n_members=CFG.v2_members, **CFG.v2_arch)
print("v2 parameters:", V.param_report(wm_, V.PIProberV2(wm_.M, wm_.d_latent)))
print("v1 parameters:", M1.n_params(M1.JEPAWorldModel()), "+", M1.n_params(M1.PIProber(32)), "prober")
del wm_

## 2. Train, seed by seed

v2 first, then v1. A finished seed is skipped, and an interrupted one resumes from its last 10-minute
checkpoint.

In [ ]:
log = []
for seed in SEEDS_THIS_SESSION:
    t0 = time.time()
    r2 = S.train_v2_seed(CFG, data, L, seed, DEVICE, grad_ckpt=GRAD_CKPT)
    r1 = S.train_v1_seed(CFG, data, L, seed, DEVICE)
    log.append({"seed": seed, "v2_cached": r2["cached"], "v1_cached": r1["cached"],
                "minutes": round((time.time() - t0) / 60, 1), "v1_extended": r1.get("extended")})
    print(f"seed {seed}: done in {(time.time() - t0) / 60:.1f} min")
    S.update_manifest(L)
display(pd.DataFrame(log))

## 3. What was trained

In [ ]:
rows = []
for kind in ("v2", "v1"):
    for seed in CFG.seeds:
        ck = L.ckpt(kind, seed)
        if not ck.exists():
            rows.append({"model": kind, "seed": seed, "status": "not trained yet"}); continue
        ex = torch.load(ck, map_location="cpu", weights_only=False)["extra"]
        rows.append({"model": kind, "seed": seed, "status": "done", "train_min": round(ex.get("train_s", 0) / 60, 1),
                     "extended": None if kind == "v2" else {k: (v or {}).get("extend") for k, v in ex["extended"].items()},
                     "min_eff_rank": min(h["eff_rank"] for h in ex["health"]) if kind == "v2" else None})
display(pd.DataFrame(rows))
for kind in ("v2", "v1"):
    p = L.model_dir(kind, CFG.seeds[0]) / "F_curves.png"
    if p.exists():
        display(Image(str(p)))